# Fine-tune fixed-annulus SNO with analytic $f=0$, $g=\alpha\cos\theta$ samples

目标：构造一批解析样本

\[
\Delta u-k^2u=0,\qquad u(r_{out},\theta)=0,\qquad \partial_n u(r_{in},\theta)=\alpha\cos\theta,
\]

其中内边界外法向为 $n=-e_r$，因此

\[
\partial_nu=-\partial_ru.
\]

本 notebook 完成四件事：

1. 解析生成 $\alpha\sim\mathcal N(1,1)$ 的 1000 个样本；
2. 检查这批样本的 FE oracle 重构误差；
3. 在冻结 FE 的前提下，用这批样本继续训练 / fine-tune Transformer；
4. 保存 fine-tuned Transformer 参数和诊断结果。

**重要审查点**：这个实验只微调 Transformer，不改 FE。若 FE 对解析样本重构误差已经很大，应先扩展或微调 FE，否则继续训练 Transformer 没有意义。

## 0. 方案合理性与注意事项

这个方案是合理的，原因是你已经定位到真实问题的困难主要来自边界条件结构分布偏移：原 PI-sampler 诱导的 $g(\theta)$ 多数不是纯 $\cos\theta$。因此构造

\[
g(\theta)=\alpha\cos\theta,
\qquad \alpha\sim\mathcal N(1,1),
\qquad f=0
\]

相当于对真实问题邻域进行 targeted adaptation。

建议：

- 使用解析解生成训练集，避免再引入数值误差；
- 固定 FE，只训练 Transformer，因为当前问题是条件输入分布偏移，不是先验解空间完全缺失；
- 先检查 FE 重构误差。若 FE oracle 的 $u$ 重构误差很小，才说明 Transformer fine-tune 是有效路径；
- 保留 validation split，避免 1000 个样本上过拟合；
- 保存 fine-tuned 参数时不要覆盖原始模型。

In [ ]:
# ============================================================
# 0. Runtime environment
# ============================================================
# 如果需要指定 GPU，请在导入 jax 前设置。
# 例如：os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import sys
import copy
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax
from flax import serialization
from scipy.special import iv, kv
from scipy.io import savemat

print("JAX devices:", jax.devices())

In [ ]:
# ============================================================
# 2. User configuration
# ============================================================
# 修改为你的 fixed-annulus SNO 工程路径。
PROJECT_DIR = "/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/annulus_sno_annulus_only_v2"

# 修改为你的输出路径和 run_name。
OUT_DIR = "/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/out"
RUN_NAME = "test"

sys.path.append(PROJECT_DIR)

from config import AnnulusConfig
from data import (
    SampleBatch,
    make_theta,
    make_annulus_grid,
    sobol_annulus_points,
    inner_boundary_coords,
    normalize_u,
    normalize_f,
    denormalize_u,
    denormalize_f,
    make_source_tokens,
    make_condition_tokens,
)
from models import FunctionEncoder
from train import (
    TrainState,
    create_fe_state,
    create_ol_state,
    load_field_normalizer,
    rl2_error,
    save_params,
)

# --------------------------
# Must match trained model architecture
# --------------------------
cfg = AnnulusConfig()
cfg.out_dir = OUT_DIR
cfg.run_name = RUN_NAME

# Geometry used by your fixed-annulus model.
cfg.r_inner = 0.2
cfg.r_outer = 1.0

# Discretization / model architecture. Must match training.
cfg.n_basis = 512
cfg.theta_size = 128
cfg.radial_size = 32
cfg.random_probe_points = 1024
cfg.pod_snapshots = 100

cfg.trunk_width = 512
cfg.trunk_depth = 5
cfg.cnn_dense_width = 1024
cfg.transformer_dim = 512
cfg.transformer_heads = 8
cfg.transformer_layers = 4
cfg.transformer_mlp_dim = 1024
cfg.seq_chunks = 32
cfg.cond_chunks = 32

# k used in analytic samples. Must be inside original training k range.
# If your original model was trained at k=1.0, keep this as 1.0.
K_VALUE = 1.0
cfg.k_min = K_VALUE
cfg.k_max = K_VALUE

# Existing parameter file names. Change if your run uses physv2 names.
FE_PARAM_NAME_CANDIDATES = [
    "fe_params.msgpack",
    "fe_params_physv2.msgpack",
]
OL_PARAM_NAME_CANDIDATES = [
    "ol_params.msgpack",
    "ol_params_physv2.msgpack",
]

# New fine-tuned model name. Do not overwrite the original model.
FINETUNED_OL_PARAM_NAME = "ol_params_alpha_cos_f0_finetuned.msgpack"

print("output_dir =", cfg.output_dir)

In [ ]:
# ============================================================
# 3. Load pretrained FE / Transformer / normalizer
# ============================================================
def find_existing_param(out_dir: Path, candidates: list[str]) -> Path:
    for name in candidates:
        path = out_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(
        "Cannot find any parameter file in candidates:\n" +
        "\n".join(str(out_dir / name) for name in candidates)
    )

key = jax.random.PRNGKey(cfg.seed + 20260625)
key, key_fe_init, key_ol_init = jax.random.split(key, 3)

normalizer = load_field_normalizer(cfg.output_dir)
print("[Normalizer]")
print("mean_u =", float(normalizer.mean_u), "std_u =", float(normalizer.std_u))
print("mean_f =", float(normalizer.mean_f), "std_f =", float(normalizer.std_f))

# FE
fe_state, fe_model = create_fe_state(cfg, key_fe_init)
fe_param_path = find_existing_param(cfg.output_dir, FE_PARAM_NAME_CANDIDATES)
fe_params = serialization.from_bytes(fe_state.params, fe_param_path.read_bytes())
fe_state = fe_state.replace(params=fe_params)
print("Loaded FE params:", fe_param_path)

# OL / Transformer
ol_state, ol_model = create_ol_state(cfg, key_ol_init)
ol_param_path = find_existing_param(cfg.output_dir, OL_PARAM_NAME_CANDIDATES)
ol_params = serialization.from_bytes(ol_state.params, ol_param_path.read_bytes())
ol_state = ol_state.replace(params=ol_params)
print("Loaded OL params:", ol_param_path)

## 4. 解析解公式

对于

\[
\Delta u-k^2u=0,
\]

且边界条件为

\[
u(R,\theta)=0,\qquad -\partial_ru(a,\theta)=\alpha\cos\theta,
\]

令 $a=r_{in}$, $R=r_{out}$。一阶角向模态解析解为

\[
u(r,\theta)=\alpha\psi(r)\cos\theta,
\]

其中

\[
\psi(r)=C\left[I_1(kr)-\beta K_1(kr)\right],
\qquad
\beta=\frac{I_1(kR)}{K_1(kR)},
\]

并由

\[
-\psi'(a)=1
\]

确定常数 $C$。因此

\[
C=-\frac{1}{k\left[I_1'(ka)-\beta K_1'(ka)\right]}.
\]

In [ ]:
# ============================================================
# 4. Analytic solution for f=0, g=alpha*cos(theta)
# ============================================================
def radial_shape_np(r, k: float, r_inner: float, r_outer: float):
    """
    Return psi(r) satisfying:
        (Delta - k^2) [psi(r) cos(theta)] = 0,
        psi(r_outer) = 0,
        -psi'(r_inner) = 1.

    Then u(r, theta; alpha) = alpha * psi(r) * cos(theta).
    """
    r = np.asarray(r, dtype=np.float64)
    a = float(r_inner)
    R = float(r_outer)
    k = float(k)

    if abs(k) < 1e-12:
        # Laplace limit for n=1: psi = A*r + B/r, psi(R)=0, -psi'(a)=1.
        # psi(r) = A * (r - R^2/r)
        # psi'(a) = A * (1 + R^2/a^2), so A = -1 / (1 + R^2/a^2)
        A = -1.0 / (1.0 + (R * R) / (a * a))
        return A * (r - (R * R) / r)

    beta = iv(1, k * R) / kv(1, k * R)

    phi = iv(1, k * r) - beta * kv(1, k * r)

    # I_1'(z) = 0.5 * (I_0(z) + I_2(z))
    I1p_a = 0.5 * (iv(0, k * a) + iv(2, k * a))

    # K_1'(z) = -0.5 * (K_0(z) + K_2(z))
    K1p_a = -0.5 * (kv(0, k * a) + kv(2, k * a))

    phi_prime_a = k * (I1p_a - beta * K1p_a)
    C = -1.0 / phi_prime_a
    return C * phi


def analytic_u_alpha_cos(coords, alpha_values, k_value, config):
    """
    coords: [N, 2]
    alpha_values: [B]
    return u: [B, N]
    """
    coords_np = np.asarray(jax.device_get(coords), dtype=np.float64)
    x = coords_np[:, 0]
    y = coords_np[:, 1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)

    psi = radial_shape_np(r, k_value, config.r_inner, config.r_outer)
    base = psi * np.cos(theta)

    alpha_np = np.asarray(alpha_values, dtype=np.float64).reshape(-1, 1)
    u = alpha_np * base.reshape(1, -1)
    return jnp.asarray(u, dtype=jnp.float32)


def make_alpha_cos_boundary_flux(alpha_values, config):
    theta = make_theta(config)[:, 0]
    base_flux = jnp.cos(theta)
    return jnp.asarray(alpha_values, dtype=jnp.float32)[:, None] * base_flux[None, :]


def check_radial_shape_sign(config, k_value):
    """Finite-difference sanity check for -psi'(r_inner)=1."""
    a = config.r_inner
    h = 1e-5
    psi_plus = radial_shape_np(np.array([a + h]), k_value, a, config.r_outer)[0]
    psi_minus = radial_shape_np(np.array([a - h]), k_value, a, config.r_outer)[0]
    dpsi = (psi_plus - psi_minus) / (2*h)
    print("Finite-diff -psi'(r_inner) ≈", -dpsi)
    print("psi(r_outer) =", radial_shape_np(np.array([config.r_outer]), k_value, a, config.r_outer)[0])

check_radial_shape_sign(cfg, K_VALUE)

In [ ]:
# ============================================================
# 5. Generate alpha dataset and analytic SampleBatch
# ============================================================
N_TOTAL = 1000
ALPHA_MEAN = 1.0
ALPHA_STD = 1.0
DATA_SEED = 20260625

# train/validation split
TRAIN_FRACTION = 0.8

rng = np.random.default_rng(DATA_SEED)
alpha_all = rng.normal(loc=ALPHA_MEAN, scale=ALPHA_STD, size=N_TOTAL).astype(np.float32)

# Optional: force one exact target sample alpha=1.0.
alpha_all[0] = 1.0

key, key_probe = jax.random.split(key)
pod_coords = make_annulus_grid(cfg)  # [Npod, 2]
probe_coords = sobol_annulus_points(
    key_probe,
    cfg.random_probe_points,
    cfg.r_inner,
    cfg.r_outer,
    cfg.dim,
)  # [Nprobe, 2]

boundary_coords_single = inner_boundary_coords(cfg)  # [Nt, 2]
boundary_coords = jnp.broadcast_to(
    boundary_coords_single[None, :, :],
    (N_TOTAL, cfg.theta_size, 2),
)

boundary_flux = make_alpha_cos_boundary_flux(alpha_all, cfg)  # [B, Nt]

u_pod = analytic_u_alpha_cos(pod_coords, alpha_all, K_VALUE, cfg)
u_probe = analytic_u_alpha_cos(probe_coords, alpha_all, K_VALUE, cfg)

f_pod = jnp.zeros_like(u_pod)
f_probe = jnp.zeros_like(u_probe)
k_values = jnp.full((N_TOTAL,), K_VALUE, dtype=jnp.float32)

alpha_batch = SampleBatch(
    boundary_coords=boundary_coords,
    boundary_flux=boundary_flux,
    pod_coords=pod_coords,
    probe_coords=probe_coords,
    u_pod=u_pod,
    f_pod=f_pod,
    u_probe=u_probe,
    f_probe=f_probe,
    k_values=k_values,
)

# Split indices
perm = rng.permutation(N_TOTAL)
n_train = int(TRAIN_FRACTION * N_TOTAL)
train_idx_np = perm[:n_train].astype(np.int32)
val_idx_np = perm[n_train:].astype(np.int32)
train_idx = jnp.asarray(train_idx_np)
val_idx = jnp.asarray(val_idx_np)

print("alpha mean/std:", float(alpha_all.mean()), float(alpha_all.std()))
print("u_pod:", u_pod.shape, "u_probe:", u_probe.shape)
print("boundary_flux:", boundary_flux.shape)
print("n_train:", n_train, "n_val:", N_TOTAL - n_train)

In [ ]:
# ============================================================
# 6. FE oracle reconstruction diagnostic
# ============================================================
def fe_reconstruct_u_f(fe_state, batch, normalizer, config):
    u_pod_norm = normalize_u(batch.u_pod, normalizer)
    f_pod_norm = normalize_f(batch.f_pod, normalizer)

    latent_u = fe_state.apply_fn(
        {"params": fe_state.params},
        u_pod_norm,
        method=FunctionEncoder.encode_u,
    )
    latent_f = fe_state.apply_fn(
        {"params": fe_state.params},
        f_pod_norm,
        method=FunctionEncoder.encode_f,
    )

    u_fe_pod_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        latent_u,
        batch.pod_coords,
        method=FunctionEncoder.reconstruct,
    )
    u_fe_probe_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        latent_u,
        batch.probe_coords,
        method=FunctionEncoder.reconstruct,
    )

    f_fe_pod_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        latent_f,
        batch.pod_coords,
        method=FunctionEncoder.reconstruct,
    )

    u_fe_pod = denormalize_u(u_fe_pod_norm, normalizer)
    u_fe_probe = denormalize_u(u_fe_probe_norm, normalizer)
    f_fe_pod = denormalize_f(f_fe_pod_norm, normalizer)

    return latent_u, latent_f, u_fe_pod, u_fe_probe, f_fe_pod

latent_u_all, latent_f_all, u_fe_pod, u_fe_probe, f_fe_pod = fe_reconstruct_u_f(
    fe_state,
    alpha_batch,
    normalizer,
    cfg,
)

err_fe_u_pod = rl2_error(u_fe_pod, alpha_batch.u_pod)
err_fe_u_probe = rl2_error(u_fe_probe, alpha_batch.u_probe)
err_fe_f_pod = rl2_error(f_fe_pod, alpha_batch.f_pod + 1e-12)  # f=0, denominator would otherwise be tiny

print("========== FE oracle reconstruction ==========")
print("mean RL2 u pod  :", float(jnp.mean(err_fe_u_pod)))
print("mean RL2 u probe:", float(jnp.mean(err_fe_u_probe)))
print("max  RL2 u probe:", float(jnp.max(err_fe_u_probe)))
print("median RL2 u probe:", float(jnp.median(err_fe_u_probe)))

plt.figure(figsize=(10, 4))
plt.hist(np.asarray(err_fe_u_probe), bins=40, alpha=0.8)
plt.xlabel("FE oracle RL2 on probe")
plt.ylabel("count")
plt.title("FE reconstruction error for analytic alpha*cos(theta), f=0 samples")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 7. Precompute Transformer inputs and baseline prediction before fine-tuning
# ============================================================
# Important: f=0 must be normalized before FE.encode_f. Do not set f_norm=0 manually.
f_tokens_all = make_source_tokens(latent_f_all, cfg)
cond_tokens_all = make_condition_tokens(alpha_batch, cfg)
target_u_latent_all = jax.lax.stop_gradient(latent_u_all)
latent_f_all = jax.lax.stop_gradient(latent_f_all)

print("f_tokens:", f_tokens_all.shape)
print("cond_tokens:", cond_tokens_all.shape)
print("target_u_latent:", target_u_latent_all.shape)


def predict_u_from_ol(ol_state, f_tokens, cond_tokens, k_values, coords, normalizer):
    pred_latent = ol_state.apply_fn(
        {"params": ol_state.params},
        f_tokens,
        cond_tokens,
        k_values,
    )
    u_pred_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        pred_latent,
        coords,
        method=FunctionEncoder.reconstruct,
    )
    u_pred = denormalize_u(u_pred_norm, normalizer)
    return pred_latent, u_pred

pred_latent_pre, u_sno_pre_probe = predict_u_from_ol(
    ol_state,
    f_tokens_all,
    cond_tokens_all,
    alpha_batch.k_values,
    alpha_batch.probe_coords,
    normalizer,
)

err_sno_pre_probe = rl2_error(u_sno_pre_probe, alpha_batch.u_probe)
err_latent_pre = rl2_error(pred_latent_pre, target_u_latent_all)

print("========== Baseline SNO before fine-tune ==========")
print("mean RL2 u probe:", float(jnp.mean(err_sno_pre_probe)))
print("median RL2 u probe:", float(jnp.median(err_sno_pre_probe)))
print("max RL2 u probe:", float(jnp.max(err_sno_pre_probe)))
print("mean latent RL2:", float(jnp.mean(err_latent_pre)))

In [ ]:
# ============================================================
# 8. Fine-tune Transformer on analytic samples
# ============================================================
FINETUNE_STEPS = 5000
BATCH_SIZE = 64
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-6
PRINT_EVERY = 200
SAVE_EVERY = 1000

schedule = optax.cosine_decay_schedule(
    init_value=FINETUNE_LR,
    decay_steps=FINETUNE_STEPS,
    alpha=0.05,
)
ft_tx = optax.adamw(schedule, weight_decay=WEIGHT_DECAY)

# Continue from pretrained Transformer params.
ol_ft_state = TrainState.create(
    apply_fn=ol_state.apply_fn,
    params=ol_state.params,
    tx=ft_tx,
)

@jax.jit
def finetune_step(state, f_tokens, cond_tokens, k_values, target_u_latent):
    def loss_fn(params):
        pred = state.apply_fn(
            {"params": params},
            f_tokens,
            cond_tokens,
            k_values,
        )
        return jnp.mean((pred - target_u_latent) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

@jax.jit
def latent_rl2_eval(state, f_tokens, cond_tokens, k_values, target_u_latent):
    pred = state.apply_fn({"params": state.params}, f_tokens, cond_tokens, k_values)
    return rl2_error(pred, target_u_latent).mean()


def take_batch(arr, idx):
    return arr[idx]

history = {
    "step": [],
    "loss": [],
    "train_latent_rl2": [],
    "val_latent_rl2": [],
    "val_u_probe_rl2": [],
}

key_train = jax.random.PRNGKey(cfg.seed + 20260626)

for step in range(FINETUNE_STEPS + 1):
    key_train, key_mb = jax.random.split(key_train)
    mb_pos = jax.random.randint(
        key_mb,
        shape=(BATCH_SIZE,),
        minval=0,
        maxval=train_idx.shape[0],
    )
    mb_idx = train_idx[mb_pos]

    ol_ft_state, loss = finetune_step(
        ol_ft_state,
        f_tokens_all[mb_idx],
        cond_tokens_all[mb_idx],
        alpha_batch.k_values[mb_idx],
        target_u_latent_all[mb_idx],
    )

    if step % PRINT_EVERY == 0:
        train_latent = latent_rl2_eval(
            ol_ft_state,
            f_tokens_all[train_idx[:min(256, train_idx.shape[0])]],
            cond_tokens_all[train_idx[:min(256, train_idx.shape[0])]],
            alpha_batch.k_values[train_idx[:min(256, train_idx.shape[0])]],
            target_u_latent_all[train_idx[:min(256, train_idx.shape[0])]],
        )
        val_latent = latent_rl2_eval(
            ol_ft_state,
            f_tokens_all[val_idx],
            cond_tokens_all[val_idx],
            alpha_batch.k_values[val_idx],
            target_u_latent_all[val_idx],
        )

        pred_latent_val = ol_ft_state.apply_fn(
            {"params": ol_ft_state.params},
            f_tokens_all[val_idx],
            cond_tokens_all[val_idx],
            alpha_batch.k_values[val_idx],
        )
        u_val_pred_norm = fe_state.apply_fn(
            {"params": fe_state.params},
            pred_latent_val,
            alpha_batch.probe_coords,
            method=FunctionEncoder.reconstruct,
        )
        u_val_pred = denormalize_u(u_val_pred_norm, normalizer)
        val_u_probe = rl2_error(u_val_pred, alpha_batch.u_probe[val_idx]).mean()

        print(
            f"[FT {step:05d}] "
            f"loss={float(loss):.3e}, "
            f"train_latent={float(train_latent):.3e}, "
            f"val_latent={float(val_latent):.3e}, "
            f"val_u_probe={float(val_u_probe):.3e}"
        )

        history["step"].append(step)
        history["loss"].append(float(loss))
        history["train_latent_rl2"].append(float(train_latent))
        history["val_latent_rl2"].append(float(val_latent))
        history["val_u_probe_rl2"].append(float(val_u_probe))

    if step > 0 and step % SAVE_EVERY == 0:
        save_params(
            ol_ft_state.params,
            cfg.output_dir / FINETUNED_OL_PARAM_NAME,
        )
        print("Saved checkpoint:", cfg.output_dir / FINETUNED_OL_PARAM_NAME)

# Final save
save_params(ol_ft_state.params, cfg.output_dir / FINETUNED_OL_PARAM_NAME)
print("Saved final fine-tuned Transformer:", cfg.output_dir / FINETUNED_OL_PARAM_NAME)

In [ ]:
# ============================================================
# 9. Evaluate after fine-tuning and compare before / after
# ============================================================
pred_latent_post, u_sno_post_probe = predict_u_from_ol(
    ol_ft_state,
    f_tokens_all,
    cond_tokens_all,
    alpha_batch.k_values,
    alpha_batch.probe_coords,
    normalizer,
)

err_sno_post_probe = rl2_error(u_sno_post_probe, alpha_batch.u_probe)
err_latent_post = rl2_error(pred_latent_post, target_u_latent_all)

print("========== Full dataset comparison ==========")
print("FE oracle mean RL2 probe:", float(jnp.mean(err_fe_u_probe)))
print("Pre  SNO mean RL2 probe:", float(jnp.mean(err_sno_pre_probe)))
print("Post SNO mean RL2 probe:", float(jnp.mean(err_sno_post_probe)))
print("Pre  latent mean RL2:", float(jnp.mean(err_latent_pre)))
print("Post latent mean RL2:", float(jnp.mean(err_latent_post)))

print("\n========== Validation subset comparison ==========")
print("Pre  val RL2 probe:", float(jnp.mean(err_sno_pre_probe[val_idx])))
print("Post val RL2 probe:", float(jnp.mean(err_sno_post_probe[val_idx])))
print("Post val max RL2 probe:", float(jnp.max(err_sno_post_probe[val_idx])))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.semilogy(history["step"], history["train_latent_rl2"], label="train latent RL2")
plt.semilogy(history["step"], history["val_latent_rl2"], label="val latent RL2")
plt.xlabel("step")
plt.ylabel("latent RL2")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(history["step"], history["val_u_probe_rl2"], label="val u probe RL2")
plt.xlabel("step")
plt.ylabel("physical RL2")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(np.asarray(err_sno_pre_probe[val_idx]), bins=40, alpha=0.6, label="pre")
plt.hist(np.asarray(err_sno_post_probe[val_idx]), bins=40, alpha=0.6, label="post")
plt.xlabel("SNO RL2 on validation probe")
plt.ylabel("count")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Before vs after fine-tuning")
plt.show()

In [ ]:
# ============================================================
# 10. Evaluate the exact target case alpha=1
# ============================================================
alpha_target = np.array([1.0], dtype=np.float32)

boundary_coords_t = jnp.broadcast_to(
    boundary_coords_single[None, :, :],
    (1, cfg.theta_size, 2),
)
boundary_flux_t = make_alpha_cos_boundary_flux(alpha_target, cfg)
u_pod_t = analytic_u_alpha_cos(pod_coords, alpha_target, K_VALUE, cfg)
u_probe_t = analytic_u_alpha_cos(probe_coords, alpha_target, K_VALUE, cfg)
f_pod_t = jnp.zeros_like(u_pod_t)
f_probe_t = jnp.zeros_like(u_probe_t)
k_t = jnp.full((1,), K_VALUE, dtype=jnp.float32)

target_batch = SampleBatch(
    boundary_coords=boundary_coords_t,
    boundary_flux=boundary_flux_t,
    pod_coords=pod_coords,
    probe_coords=probe_coords,
    u_pod=u_pod_t,
    f_pod=f_pod_t,
    u_probe=u_probe_t,
    f_probe=f_probe_t,
    k_values=k_t,
)

u_pod_t_norm = normalize_u(target_batch.u_pod, normalizer)
f_pod_t_norm = normalize_f(target_batch.f_pod, normalizer)
latent_u_t = fe_state.apply_fn({"params": fe_state.params}, u_pod_t_norm, method=FunctionEncoder.encode_u)
latent_f_t = fe_state.apply_fn({"params": fe_state.params}, f_pod_t_norm, method=FunctionEncoder.encode_f)
f_tokens_t = make_source_tokens(latent_f_t, cfg)
cond_tokens_t = make_condition_tokens(target_batch, cfg)

# FE oracle target
u_fe_t_norm = fe_state.apply_fn({"params": fe_state.params}, latent_u_t, probe_coords, method=FunctionEncoder.reconstruct)
u_fe_t = denormalize_u(u_fe_t_norm, normalizer)
err_fe_t = rl2_error(u_fe_t, target_batch.u_probe)

# pre/post SNO target
_, u_pre_t = predict_u_from_ol(ol_state, f_tokens_t, cond_tokens_t, k_t, probe_coords, normalizer)
_, u_post_t = predict_u_from_ol(ol_ft_state, f_tokens_t, cond_tokens_t, k_t, probe_coords, normalizer)
err_pre_t = rl2_error(u_pre_t, target_batch.u_probe)
err_post_t = rl2_error(u_post_t, target_batch.u_probe)

print("========== Exact target alpha=1 ==========")
print("FE oracle RL2 probe:", float(err_fe_t[0]))
print("Pre  SNO RL2 probe:", float(err_pre_t[0]))
print("Post SNO RL2 probe:", float(err_post_t[0]))

# quick visualization on pod grid
_, u_pre_t_pod = predict_u_from_ol(ol_state, f_tokens_t, cond_tokens_t, k_t, pod_coords, normalizer)
_, u_post_t_pod = predict_u_from_ol(ol_ft_state, f_tokens_t, cond_tokens_t, k_t, pod_coords, normalizer)

Nr, Nt = cfg.radial_size, cfg.theta_size
coords_np = np.asarray(jax.device_get(pod_coords))
X = coords_np[:, 0].reshape(Nr, Nt)
Y = coords_np[:, 1].reshape(Nr, Nt)
U_true = np.asarray(jax.device_get(target_batch.u_pod[0])).reshape(Nr, Nt)
U_pre = np.asarray(jax.device_get(u_pre_t_pod[0])).reshape(Nr, Nt)
U_post = np.asarray(jax.device_get(u_post_t_pod[0])).reshape(Nr, Nt)

vmin = min(U_true.min(), U_pre.min(), U_post.min())
vmax = max(U_true.max(), U_pre.max(), U_post.max())

plt.figure(figsize=(15, 4))
for i, (Z, title) in enumerate([
    (U_true, "true alpha=1"),
    (U_pre, f"pre SNO, RL2={float(err_pre_t[0]):.3e}"),
    (U_post, f"post SNO, RL2={float(err_post_t[0]):.3e}"),
]):
    plt.subplot(1, 3, i + 1)
    plt.pcolormesh(X, Y, Z, shading="auto", vmin=vmin, vmax=vmax)
    plt.axis("equal")
    plt.axis("off")
    plt.title(title)
    plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 11. Export diagnostics to .mat / .npz
# ============================================================
EXPORT_GRID_FIELDS = True

out_mat = cfg.output_dir / "alpha_cos_f0_finetune_diagnostics.mat"
out_npz = cfg.output_dir / "alpha_cos_f0_finetune_history.npz"

mat_data = {
    "alpha": np.asarray(alpha_all)[:, None],
    "train_idx": train_idx_np[:, None] + 1,  # MATLAB 1-based
    "val_idx": val_idx_np[:, None] + 1,
    "pod_coords": np.asarray(jax.device_get(pod_coords)),
    "probe_coords": np.asarray(jax.device_get(probe_coords)),
    "boundary_coords": np.asarray(jax.device_get(boundary_coords)),
    "boundary_flux": np.asarray(jax.device_get(boundary_flux)),
    "k_values": np.asarray(jax.device_get(k_values))[:, None],
    "err_fe_u_pod": np.asarray(jax.device_get(err_fe_u_pod))[:, None],
    "err_fe_u_probe": np.asarray(jax.device_get(err_fe_u_probe))[:, None],
    "err_sno_pre_probe": np.asarray(jax.device_get(err_sno_pre_probe))[:, None],
    "err_sno_post_probe": np.asarray(jax.device_get(err_sno_post_probe))[:, None],
    "err_latent_pre": np.asarray(jax.device_get(err_latent_pre))[:, None],
    "err_latent_post": np.asarray(jax.device_get(err_latent_post))[:, None],
    "history_step": np.asarray(history["step"])[:, None],
    "history_loss": np.asarray(history["loss"])[:, None],
    "history_train_latent_rl2": np.asarray(history["train_latent_rl2"])[:, None],
    "history_val_latent_rl2": np.asarray(history["val_latent_rl2"])[:, None],
    "history_val_u_probe_rl2": np.asarray(history["val_u_probe_rl2"])[:, None],
    "target_alpha1_err_fe_probe": np.array([[float(err_fe_t[0])]]),
    "target_alpha1_err_pre_probe": np.array([[float(err_pre_t[0])]]),
    "target_alpha1_err_post_probe": np.array([[float(err_post_t[0])]]),
    "r_inner": np.array([[cfg.r_inner]]),
    "r_outer": np.array([[cfg.r_outer]]),
    "k_value": np.array([[K_VALUE]]),
    "radial_size": np.array([[cfg.radial_size]]),
    "theta_size": np.array([[cfg.theta_size]]),
}

if EXPORT_GRID_FIELDS:
    Nr, Nt = cfg.radial_size, cfg.theta_size
    mat_data.update({
        "x_grid": X,
        "y_grid": Y,
        "u_true_grid": np.asarray(jax.device_get(alpha_batch.u_pod)).reshape(N_TOTAL, Nr, Nt),
        "u_fe_grid": np.asarray(jax.device_get(u_fe_pod)).reshape(N_TOTAL, Nr, Nt),
    })

savemat(out_mat, mat_data, do_compression=True)
np.savez(
    out_npz,
    **{k: np.asarray(v) for k, v in history.items()},
)

print("Saved diagnostics MAT:", out_mat)
print("Saved history NPZ:", out_npz)
print("Saved fine-tuned model:", cfg.output_dir / FINETUNED_OL_PARAM_NAME)